# 04 — Carga da Camada Gold

Este notebook lê a Silver e produz datasets analíticos prontos para:
- **Dashboards** e relatórios executivos
- **Análises estatísticas** de desigualdade educacional
- **Treinamento de modelos de ML** (predição de alfabetização)

**Datasets produzidos**:
1. `tc02_indicador_por_municipio` — Dataset principal com contexto geográfico e social completo
2. `tc02_metas_vs_resultados` — Comparação entre metas e resultados reais por UF/ano
3. `tc02_evolucao_temporal` — Série histórica do indicador por nível geográfico
4. `tc02_ranking_municipios` — Ranking de municípios no ano mais recente

**Aplicação em IA**: a camada Gold alimenta modelos preditivos de alfabetização
e análises de cluster de vulnerabilidade educacional.

In [0]:
from pyspark.sql import functions as F
from pyspark.sql import Window

# Verifica pré-requisito
try:
    spark.read.table("silver.tc02_integrado").limit(1).count()
except Exception as err:
    raise ValueError(
        "Tabela 'silver.tc02_integrado' não encontrada. Execute 03_carga_camada_silver.py."
    ) from err

df_silver    = spark.read.table("silver.tc02_integrado")
df_meta_br   = spark.read.table("silver.tc02_meta_brasil")
df_silver_uf = spark.read.table("silver.tc02_dim_uf")

print(f"Silver carregada: {df_silver.count()} registros")

## Gold 1: Indicador por Município (Dataset Principal)

Dataset completo com contexto geográfico, social e comparativo com metas.
Inclui classificação por faixa de desempenho e gap em relação à meta.

In [0]:
df_gold_indicador_municipio = (
    df_silver
    .select(
        "id_municipio",
        "nome_municipio",
        "sigla_uf",
        "nome_uf",
        "regiao",
        "capital",
        "populacao_estimada",
        "ano",
        "total_alunos_2o_ano",
        "alunos_alfabetizados",
        "indicador_crianca_alfabetizada",
        "meta_nacional",
        "meta_uf",
        "ponto_corte_saeb",
    )
    .withColumn(
        "meta_referencia",
        F.coalesce(F.col("meta_uf"), F.col("meta_nacional"))
    )
    .withColumn(
        "gap_meta",
        F.round(F.col("meta_referencia") - F.col("indicador_crianca_alfabetizada"), 2)
    )
    .withColumn(
        "atingiu_meta",
        F.col("indicador_crianca_alfabetizada") >= F.col("meta_referencia")
    )
    .withColumn(
        "faixa_desempenho",
        F.when(F.col("indicador_crianca_alfabetizada") >= 70, "Alta (≥70%)")
         .when(F.col("indicador_crianca_alfabetizada") >= 55, "Média (55–70%)")
         .when(F.col("indicador_crianca_alfabetizada") >= 40, "Baixa (40–55%)")
         .otherwise("Crítica (<40%)")
    )
    .withColumn(
        "categoria_capital",
        F.when(F.col("capital"), "Capital").otherwise("Interior")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Indicador Município: {df_gold_indicador_municipio.count()} registros")
display(
    df_gold_indicador_municipio
    .orderBy("sigla_uf", "ano")
    .select("nome_municipio", "sigla_uf", "regiao", "ano",
            "indicador_crianca_alfabetizada", "meta_referencia", "gap_meta", "faixa_desempenho")
    .limit(20)
)

## Gold 2: Metas vs Resultados por UF

Agrega os resultados por UF e ano, calcula o gap em relação às metas
e classifica a situação de cada estado.

In [0]:
df_gold_metas_resultados = (
    df_silver
    .groupBy("sigla_uf", "nome_uf", "regiao", "ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio_uf"),
        F.round(F.min("indicador_crianca_alfabetizada"), 2).alias("indicador_min_uf"),
        F.round(F.max("indicador_crianca_alfabetizada"), 2).alias("indicador_max_uf"),
        F.round(F.stddev("indicador_crianca_alfabetizada"), 2).alias("desvio_padrao_uf"),
        F.first("meta_uf").alias("meta_uf"),
        F.first("meta_nacional").alias("meta_nacional"),
        F.sum("total_alunos_2o_ano").alias("total_alunos_uf"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados_uf"),
        F.countDistinct("id_municipio").alias("qtd_municipios"),
    )
    .withColumn("meta_referencia", F.coalesce(F.col("meta_uf"), F.col("meta_nacional")))
    .withColumn(
        "gap_meta",
        F.round(F.col("meta_referencia") - F.col("indicador_medio_uf"), 2)
    )
    .withColumn("atingiu_meta", F.col("indicador_medio_uf") >= F.col("meta_referencia"))
    .withColumn(
        "situacao",
        F.when(F.col("indicador_medio_uf") >= F.col("meta_referencia"), "Meta Atingida")
         .when(F.col("gap_meta") <= 5.0, "Próximo da Meta (≤5pp)")
         .otherwise("Abaixo da Meta")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
    .orderBy("ano", "gap_meta")
)

print(f"Gold Metas vs Resultados: {df_gold_metas_resultados.count()} registros")
display(df_gold_metas_resultados.select(
    "sigla_uf", "nome_uf", "regiao", "ano", "indicador_medio_uf",
    "meta_referencia", "gap_meta", "situacao", "qtd_municipios"
).limit(30))

## Gold 3: Evolução Temporal

Série histórica do indicador em dois níveis: **Brasil** e **por UF**.
Inclui variação ano a ano (pontos percentuais) para análise de tendência.

In [0]:
janela_evolucao = Window.partitionBy("nivel", "referencia").orderBy("ano")

df_evolucao_brasil = (
    df_silver
    .groupBy("ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio"),
        F.sum("total_alunos_2o_ano").alias("total_alunos"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados"),
        F.first("meta_nacional").alias("meta"),
    )
    .withColumn("nivel", F.lit("BRASIL"))
    .withColumn("referencia", F.lit("BRASIL"))
    .withColumn("nome_referencia", F.lit("Brasil"))
    .withColumn("regiao", F.lit("Nacional"))
)

df_evolucao_uf = (
    df_silver
    .groupBy("sigla_uf", "nome_uf", "regiao", "ano")
    .agg(
        F.round(F.avg("indicador_crianca_alfabetizada"), 2).alias("indicador_medio"),
        F.sum("total_alunos_2o_ano").alias("total_alunos"),
        F.sum("alunos_alfabetizados").alias("total_alfabetizados"),
        F.first("meta_uf").alias("meta"),
    )
    .withColumn("nivel", F.lit("UF"))
    .withColumnRenamed("sigla_uf", "referencia")
    .withColumnRenamed("nome_uf", "nome_referencia")
)

colunas_comuns = ["ano", "nivel", "referencia", "nome_referencia", "regiao",
                  "indicador_medio", "total_alunos", "total_alfabetizados", "meta"]

df_gold_evolucao = (
    df_evolucao_brasil.select(*colunas_comuns)
    .unionByName(df_evolucao_uf.select(*colunas_comuns))
    .withColumn(
        "variacao_pp",
        F.round(
            F.col("indicador_medio") - F.lag("indicador_medio", 1).over(janela_evolucao), 2
        )
    )
    .withColumn(
        "tendencia",
        F.when(F.col("variacao_pp") > 0, "Melhora")
         .when(F.col("variacao_pp") < 0, "Piora")
         .otherwise("Estável")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Evolução Temporal: {df_gold_evolucao.count()} registros")
display(
    df_gold_evolucao
    .filter(F.col("nivel") == "BRASIL")
    .orderBy("ano")
    .select("ano", "nivel", "referencia", "indicador_medio", "meta", "variacao_pp", "tendencia")
)

## Gold 4: Ranking de Municípios

Ranking nacional e regional de municípios no ano mais recente.
Útil para identificar boas práticas (top performers) e municípios vulneráveis.

In [0]:
ano_mais_recente = df_silver.agg(F.max("ano")).first()[0]

janela_nacional = Window.orderBy(F.col("indicador_crianca_alfabetizada").desc())
janela_regiao   = Window.partitionBy("regiao").orderBy(F.col("indicador_crianca_alfabetizada").desc())
janela_uf       = Window.partitionBy("sigla_uf").orderBy(F.col("indicador_crianca_alfabetizada").desc())

df_gold_ranking = (
    df_silver
    .filter(F.col("ano") == ano_mais_recente)
    .select(
        "id_municipio", "nome_municipio", "sigla_uf", "nome_uf", "regiao",
        "capital", "populacao_estimada",
        "indicador_crianca_alfabetizada",
        "total_alunos_2o_ano", "alunos_alfabetizados",
        "meta_nacional", "meta_uf",
    )
    .withColumn("ranking_nacional", F.rank().over(janela_nacional))
    .withColumn("ranking_regiao",   F.rank().over(janela_regiao))
    .withColumn("ranking_uf",       F.rank().over(janela_uf))
    .withColumn(
        "quartil_nacional",
        F.when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.25, "Q1 - Top 25%")
         .when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.50, "Q2 - 25-50%")
         .when(F.col("ranking_nacional") <= F.count("*").over(Window.orderBy(F.lit(1))) * 0.75, "Q3 - 50-75%")
         .otherwise("Q4 - Bottom 25%")
    )
    .withColumn("_data_processamento_gold", F.current_timestamp())
)

print(f"Gold Ranking Municípios (ano {ano_mais_recente}): {df_gold_ranking.count()} registros")
display(
    df_gold_ranking
    .orderBy("ranking_nacional")
    .select("ranking_nacional", "nome_municipio", "sigla_uf", "regiao",
            "indicador_crianca_alfabetizada", "capital", "quartil_nacional")
    .limit(20)
)

## Escrita na Camada Gold

In [0]:
spark.sql("CREATE SCHEMA IF NOT EXISTS gold")

tabelas_gold = {
    "gold.tc02_indicador_por_municipio": df_gold_indicador_municipio,
    "gold.tc02_metas_vs_resultados":     df_gold_metas_resultados,
    "gold.tc02_evolucao_temporal":       df_gold_evolucao,
    "gold.tc02_ranking_municipios":      df_gold_ranking,
}

for nome, df in tabelas_gold.items():
    (
        df.write
        .format("delta")
        .mode("overwrite")
        .option("overwriteSchema", "true")
        .saveAsTable(nome)
    )

print("Tabelas Gold criadas com sucesso:")
for nome in tabelas_gold:
    print(f"  - {nome} => {spark.read.table(nome).count()} linhas")

## Resumo da Camada Gold

In [0]:
print("=" * 60)
print("SUMÁRIO EXECUTIVO — PIPELINE BATCH CONCLUÍDO")
print("=" * 60)

total_municipios = spark.read.table("gold.tc02_indicador_por_municipio").select("id_municipio").distinct().count()
anos_disponiveis = sorted([r.ano for r in spark.read.table("gold.tc02_evolucao_temporal").filter(F.col("nivel") == "BRASIL").select("ano").collect()])
indicador_br = spark.read.table("gold.tc02_evolucao_temporal").filter(
    (F.col("nivel") == "BRASIL") & (F.col("ano") == max(anos_disponiveis))
).first()

print(f"\nMunicípios cobertos:          {total_municipios}")
print(f"Período analisado:            {min(anos_disponiveis)}–{max(anos_disponiveis)}")
print(f"Indicador Brasil ({max(anos_disponiveis)}):   {indicador_br['indicador_medio']}%")

acima_meta = spark.read.table("gold.tc02_metas_vs_resultados").filter(
    (F.col("ano") == max(anos_disponiveis)) & F.col("atingiu_meta")
).count()
total_ufs = spark.read.table("gold.tc02_metas_vs_resultados").filter(
    F.col("ano") == max(anos_disponiveis)
).count()
print(f"UFs que atingiram a meta:     {acima_meta}/{total_ufs}")
print(f"\nTabelas Gold disponíveis:")
for nome in tabelas_gold:
    print(f"  - {nome}")

print("\nPróximo passo: executar 05_streaming_simulacao.py")